# Patient Mortality Modeling
Recovered final logistic-regression experiment. The supplied model-input CSV was confirmed by the project owner; place it at `data/testing_h.csv`.
This preserves the historical procedure, including outcome-informed thresholds calculated before splitting. Metrics in the README are historical report results, not results of a new run.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.preprocessing import StandardScaler

# Load the dataset
from pathlib import Path

# Run this notebook from the notebooks directory.
data_path = Path("../data/testing_h.csv")
data = pd.read_csv(data_path)

# Step 1: Define thresholds for extreme values
variables = [
    'norepinephrine_dosage', 'total_creatinine_value', 'blood_pressure_mean',
    'mean_glucose', 'mean_potassium', 'mean_sodium',
    'mean_arterial_co2', 'mean_ph_arterial', 'total_ventilation_duration', 'hrv_range',
    'mean_sao2', 'min_sao2', 'std_sao2', 'time_below_90','mean_ph_arterial'
]

# Define extreme thresholds for deceased patients
deceased_data = data[data['hospital_expire_flag'] == 1]
extreme_thresholds = {var: deceased_data[var].quantile(0.90) for var in variables}
low_thresholds = {var: deceased_data[var].quantile(0.10) for var in variables}

# Create binary flags for extreme values
for var in variables:
    if var in ['mean_ph_arterial', 'mean_arterial_co2', 'min_sao2','blood_pressure_mean','mean_sao2','mean_ph_arterial']:  # Low is critical
        flag_name = f'low_{var}'
        data[flag_name] = (data[var] < low_thresholds[var]).astype(int)
    else:  # High is critical
        flag_name = f'high_{var}'
        data[flag_name] = (data[var] > extreme_thresholds[var]).astype(int)

# Step 2: Create new interaction features
data['glucose_potassium_interaction'] = data['mean_glucose'] * data['mean_potassium']
data['norepinephrine_pressure_interaction'] = data['high_norepinephrine_dosage'] * data['blood_pressure_mean']
data['sao2_variability_interaction'] = data['std_sao2'] * data['time_below_90']
data['metabolic_stress(mean_sao2 X mean_glucose)'] = data['mean_sao2'] * data['mean_glucose']

# Step 3: Define features and target
features = [
    'high_norepinephrine_dosage',
    'blood_pressure_mean', 'high_mean_glucose', 'high_mean_potassium', 'high_mean_sodium',
    'high_total_ventilation_duration',
    'high_hrv_range',
    'glucose_potassium_interaction','mean_sao2', 'min_sao2', 'std_sao2', 'time_below_90',
    'metabolic_stress(mean_sao2 X mean_glucose)','mean_ph_arterial'
]
target = 'hospital_expire_flag'

X = data[features]
y = data[target]

# Step 4: Split the dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)


from imblearn.over_sampling import SMOTE
# Step 4.1: Apply SMOTE to oversample the training set
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)


# Step 5: Standardize the features
scaler = StandardScaler()
X_train_resampled = scaler.fit_transform(X_train_resampled)
X_test = scaler.transform(X_test)


# Step 6: Train a logistic regression model
logistic_model = LogisticRegression(max_iter=1000, class_weight='balanced')
logistic_model.fit(X_train_resampled, y_train_resampled)


# Step 7: Evaluate the model
y_pred = logistic_model.predict(X_test)
y_proba = logistic_model.predict_proba(X_test)[:, 1]


# Performance metrics
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

# Feature importance
coefficients = pd.DataFrame({
    'Feature': features,
    'Coefficient': logistic_model.coef_[0]
}).sort_values(by='Coefficient', ascending=False)
print("\nFeature Coefficients:\n", coefficients)

# Plot feature coefficients
plt.figure(figsize=(10, 6))
sns.barplot(x='Coefficient', y='Feature', data=coefficients, palette='coolwarm')
plt.title("Feature Importance")
plt.tight_layout()
plt.show()

# Step 8: Correlation heatmap
correlation_matrix = data[features + [target]].corr()
plt.figure(figsize=(12, 8))
sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    cbar=True,
    square=True
)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()



from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, roc_auc_score, accuracy_score

# Step 6: Train a linear regression model
linear_model = LinearRegression()
linear_model.fit(X_train_resampled, y_train_resampled)

# Step 7: Predict using the linear regression model
y_lr_pred = linear_model.predict(X_test)

# Step 8: Convert predictions to binary outcomes
threshold = 0.5  # Standard threshold for binary classification
y_lr_binary_pred = (y_lr_pred >= threshold).astype(int)

# Step 9: Evaluate the linear regression model
mse = mean_squared_error(y_test, y_lr_pred)
r2 = r2_score(y_test, y_lr_pred)
roc_auc = roc_auc_score(y_test, y_lr_pred)

# Print evaluation metrics
print("Linear Regression Mean Squared Error:", mse)
print("Linear Regression R^2 Score:", r2)
print("Linear Regression ROC-AUC:", roc_auc)
print("Linear Regression Accuracy (binary threshold = 0.5):", accuracy_score(y_test, y_lr_binary_pred))

# Step 10: Analyze coefficients
lr_coefficients = pd.DataFrame({
    'Feature': features,
    'Coefficient': linear_model.coef_
}).sort_values(by='Coefficient', ascending=False)

print("\nLinear Regression Coefficients:\n", lr_coefficients)

# Plot feature coefficients
plt.figure(figsize=(10, 6))
sns.barplot(x='Coefficient', y='Feature', data=lr_coefficients, palette='coolwarm')
plt.title("Linear Regression Feature Importance")
plt.tight_layout()
plt.show()
